# RF Best Tune: DIFF + LOGDIFF Volume + ABS Elo

Evaluate the tuned Random Forest setup with the `DIFF + LOGDIFF volume + ABS Elo` feature method using five random stratified 90/10 splits.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline


RANDOM_STATE = 42
N_RANDOM_SPLITS = 5
TEST_SIZE = 0.10

RF_PARAMS = {
    "n_estimators": 600,
    "max_depth": None,
    "max_features": "sqrt",
    "min_samples_leaf": 8,
    "min_samples_split": 2,
    "bootstrap": True,
    "criterion": "gini",
    "class_weight": None,
}

data_path = Path("../data_clean/pre_fight_career_stats.csv")
if not data_path.exists():
    data_path = Path("data_clean/pre_fight_career_stats.csv")

df = pd.read_csv(data_path)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df[df["winner_side"].isin(["R", "B"])].copy()
df = df.dropna(subset=["date", "winner_side"]).reset_index(drop=True)

print("Loaded:", data_path)
print("Shape:", df.shape)
print(df["winner_side"].value_counts())
df.head()

Loaded: ..\data_clean\pre_fight_career_stats.csv
Shape: (3040, 121)
winner_side
R    1825
B    1215
Name: count, dtype: int64


,fight_id,date,R_fighter,B_fighter,R_age,B_age,winner,winner_side,win_by,R_elo,...,B_DISTANCE_absorbed_per_min,B_DISTANCE_opp_att_per_min,B_CLINCH_land_per_min,B_CLINCH_att_per_min,B_CLINCH_absorbed_per_min,B_CLINCH_opp_att_per_min,B_GROUND_land_per_min,B_GROUND_att_per_min,B_GROUND_absorbed_per_min,B_GROUND_opp_att_per_min
0,8644,1995-12-16,Dan Severn,Oleg Taktarov,37.522245,28.306639,Dan Severn,R,Decision - Unanimous,1573.348793,...,0.581583,2.132472,0.038772,0.135703,0.213247,0.465267,0.000000,0.000000,0.678514,1.124394
1,8645,1995-12-16,Oleg Taktarov,Dave Beneteau,28.306639,NaN,Oleg Taktarov,R,Submission,1558.827392,...,0.000000,0.000000,0.920716,2.455243,0.153453,0.613811,3.222506,5.217391,0.153453,0.153453
2,8647,1995-12-16,Dan Severn,Paul Varelans,37.522245,26.245038,Dan Severn,R,Submission,1593.683396,...,1.912568,3.606557,0.491803,0.765027,0.327869,1.366120,0.327869,0.327869,0.928962,1.475410
3,8628,1996-05-17,Dan Severn,Ken Shamrock,37.941136,32.262834,Dan Severn,R,Decision - Split,1609.448071,...,0.205781,0.911318,0.235179,0.293974,0.146987,0.235179,1.028907,1.499265,0.146987,0.205781
4,8606,1996-12-07,Ken Shamrock,Brian Johnston,32.821355,28.361396,Ken Shamrock,R,Submission,1562.685802,...,0.203390,0.508475,1.220339,1.830508,0.508475,0.711864,0.711864,1.118644,1.627119,1.830508


## Balance Side Labels

Randomly flip red/blue orientation so target labels are as close to 50/50 as possible within each year.

In [2]:
def find_red_blue_pairs(columns):
    pairs = []
    for col in columns:
        if col.startswith("R_"):
            b_col = "B_" + col[2:]
            if b_col in columns:
                pairs.append((col, b_col))
    return pairs


def swap_red_blue_rows(data, swap_mask):
    balanced = data.copy()
    pairs = find_red_blue_pairs(balanced.columns)
    r_cols = [r for r, _ in pairs]
    b_cols = [b for _, b in pairs]

    r_values = balanced.loc[swap_mask, r_cols].to_numpy(copy=True)
    b_values = balanced.loc[swap_mask, b_cols].to_numpy(copy=True)
    balanced.loc[swap_mask, r_cols] = b_values
    balanced.loc[swap_mask, b_cols] = r_values

    balanced.loc[swap_mask, "winner_side"] = balanced.loc[swap_mask, "winner_side"].map({"R": "B", "B": "R"})
    return balanced


def randomly_balance_winner_side(data, random_state=42, balance_by_year=True):
    rng = np.random.default_rng(random_state)
    balanced = data.copy().reset_index(drop=True)

    desired_side = pd.Series(index=balanced.index, dtype="object")
    groups = balanced["date"].dt.year if balance_by_year else pd.Series(0, index=balanced.index)

    for _, group_idx in balanced.groupby(groups).groups.items():
        group_idx = np.array(list(group_idx))
        n_rows = len(group_idx)
        group_sides = ["R"] * (n_rows // 2) + ["B"] * (n_rows // 2)

        if n_rows % 2 == 1:
            group_sides.append(rng.choice(["R", "B"]))

        group_sides = np.array(group_sides)
        rng.shuffle(group_sides)
        desired_side.loc[group_idx] = group_sides

    swap_mask = balanced["winner_side"].to_numpy() != desired_side.to_numpy()
    balanced = swap_red_blue_rows(balanced, swap_mask)
    balanced["target"] = (balanced["winner_side"] == "R").astype(int)
    return balanced


balanced_df = randomly_balance_winner_side(df, random_state=RANDOM_STATE)

print("Original target distribution:")
print(df["winner_side"].value_counts())
print("\nBalanced target distribution:")
print(balanced_df["winner_side"].value_counts())

balanced_df.sort_values("date").groupby(balanced_df["date"].dt.year)["winner_side"].value_counts(normalize=True).unstack().tail(10)

Original target distribution:
winner_side
R    1825
B    1215
Name: count, dtype: int64

Balanced target distribution:
winner_side
R    1523
B    1517
Name: count, dtype: int64


winner_side,B,R
date,,
2017,0.497110,0.502890
2018,0.500000,0.500000
2019,0.497207,0.502793
2020,0.502924,0.497076
2021,0.497608,0.502392
2022,0.500000,0.500000
2023,0.500000,0.500000
2024,0.500000,0.500000
2025,0.502075,0.497925


## Feature Method

Use `DIFF_...` for every numeric R/B pair, `LOGDIFF_...` for volume/count/rate features, and `ABS_DIFF_elo` for Elo gap magnitude.

In [3]:
def is_logdiff_candidate(suffix):
    skip_terms = ["pct", "acc", "def", "quickness", "elo", "age"]
    return not any(term in suffix.lower() for term in skip_terms)


def add_matchup_features(data):
    new_features = {}

    for r_col, b_col in find_red_blue_pairs(data.columns):
        suffix = r_col[2:]
        if not (pd.api.types.is_numeric_dtype(data[r_col]) and pd.api.types.is_numeric_dtype(data[b_col])):
            continue

        diff_col = f"DIFF_{suffix}"
        new_features[diff_col] = data[r_col] - data[b_col]

        if is_logdiff_candidate(suffix):
            r_values = data[r_col].clip(lower=0)
            b_values = data[b_col].clip(lower=0)
            new_features[f"LOGDIFF_{suffix}"] = np.log1p(r_values) - np.log1p(b_values)

        if suffix == "elo":
            new_features["ABS_DIFF_elo"] = new_features[diff_col].abs()

    return pd.concat([data.copy(), pd.DataFrame(new_features, index=data.index)], axis=1)


model_df = add_matchup_features(balanced_df)
y = model_df["target"]

AGE_CENTER = model_df[["R_age", "B_age"]].stack().median()
model_df["DIFF_age_sq"] = model_df["R_age"] ** 2 - model_df["B_age"] ** 2
model_df["R_age_from_prime"] = (model_df["R_age"] - AGE_CENTER).abs()
model_df["B_age_from_prime"] = (model_df["B_age"] - AGE_CENTER).abs()
model_df["DIFF_age_from_prime"] = model_df["R_age_from_prime"] - model_df["B_age_from_prime"]
model_df["DIFF_age_centered_sq"] = (
    (model_df["R_age"] - AGE_CENTER) ** 2
    - (model_df["B_age"] - AGE_CENTER) ** 2
)
model_df["ABS_DIFF_age_centered_sq"] = model_df["DIFF_age_centered_sq"].abs()

diff_cols = [col for col in model_df.columns if col.startswith("DIFF_")]
logdiff_cols = [col for col in model_df.columns if col.startswith("LOGDIFF_")]
abs_elo_cols = [col for col in ["ABS_DIFF_elo"] if col in model_df.columns]

feature_cols = list(dict.fromkeys(diff_cols + logdiff_cols + abs_elo_cols))
X = model_df[feature_cols].replace([np.inf, -np.inf], np.nan)

print("Rows:", len(model_df))
print("Feature count:", len(feature_cols))
feature_cols[:20]

Rows: 3040
Feature count: 102


['DIFF_age',
 'DIFF_elo',
 'DIFF_total_fights',
 'DIFF_total_fight_time',
 'DIFF_fight_quickness',
 'DIFF_win_DEC_pct',
 'DIFF_win_KO_TKO_pct',
 'DIFF_win_SUB_pct',
 'DIFF_loss_DEC_pct',
 'DIFF_loss_KO_TKO_pct',
 'DIFF_loss_SUB_pct',
 'DIFF_TOTAL_STR_acc',
 'DIFF_SIG_STR_acc',
 'DIFF_TD_acc',
 'DIFF_CTRL_pct',
 'DIFF_KD_per_min',
 'DIFF_SUB_ATT_per_min',
 'DIFF_REV_per_min',
 'DIFF_SIG_STR_land_per_min',
 'DIFF_SIG_STR_att_per_min']

## Age Feature Comparison

Compare different age encodings while keeping the rest of the feature method fixed.

In [8]:
def make_age_test_model(random_state):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    **RF_PARAMS,
                    n_jobs=-1,
                    random_state=random_state,
                ),
            ),
        ]
    )


age_variant_cols = [
    "DIFF_age",
    "DIFF_age_sq",
    "R_age_from_prime",
    "B_age_from_prime",
    "DIFF_age_from_prime",
    "DIFF_age_centered_sq",
    "ABS_DIFF_age_centered_sq",
]

base_non_age_cols = [
    col for col in feature_cols
    if col not in age_variant_cols
]

age_feature_sets = {
    "DIFF_age": ["DIFF_age"],
    "DIFF_age_sq": ["DIFF_age_sq"],
    "DIFF_age_from_prime": ["DIFF_age_from_prime"],
    "age_from_prime features": [
        "R_age_from_prime",
        "B_age_from_prime",
        "DIFF_age_from_prime",
    ],
    "DIFF_age_centered_sq": ["DIFF_age_centered_sq"],
    "abs_centered_sq": ["ABS_DIFF_age_centered_sq"],
}

age_splitter = StratifiedShuffleSplit(
    n_splits=N_RANDOM_SPLITS,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)
age_splits = list(age_splitter.split(model_df[base_non_age_cols], y))

age_results = []
for feature_set_name, age_cols in age_feature_sets.items():
    cols = list(dict.fromkeys(base_non_age_cols + age_cols))
    X_age = model_df[cols].replace([np.inf, -np.inf], np.nan)

    accuracies = []
    aucs = []
    for split_num, (train_idx, test_idx) in enumerate(age_splits, start=1):
        model = make_age_test_model(RANDOM_STATE + split_num)
        model.fit(X_age.iloc[train_idx], y.iloc[train_idx])
        pred = model.predict(X_age.iloc[test_idx])
        proba = model.predict_proba(X_age.iloc[test_idx])[:, 1]
        accuracies.append(accuracy_score(y.iloc[test_idx], pred))
        aucs.append(roc_auc_score(y.iloc[test_idx], proba))

    age_results.append({
        "age_feature_set": feature_set_name,
        "age_center": AGE_CENTER,
        "n_features": len(cols),
        "mean_accuracy": np.mean(accuracies),
        "std_accuracy": np.std(accuracies),
        "mean_auc": np.mean(aucs),
        "std_auc": np.std(aucs),
        "split_accuracies": accuracies,
    })

age_feature_results = pd.DataFrame(age_results).sort_values(
    ["mean_accuracy", "mean_auc"],
    ascending=False,
).reset_index(drop=True)

age_feature_results

,age_feature_set,age_center,n_features,mean_accuracy,std_accuracy,mean_auc,std_auc,split_accuracies
0,DIFF_age_centered_sq,32.093087,99,0.624342,0.028404,0.646122,0.033404,"[0.5986842105263158, 0.5921052631578947, 0.618..."
1,age_from_prime features,32.093087,101,0.621053,0.027237,0.646434,0.033251,"[0.6085526315789473, 0.5986842105263158, 0.598..."
2,DIFF_age_from_prime,32.093087,99,0.621053,0.028020,0.646208,0.035878,"[0.5921052631578947, 0.5921052631578947, 0.625..."
3,DIFF_age,32.093087,99,0.620395,0.036250,0.668508,0.042154,"[0.5789473684210527, 0.5921052631578947, 0.611..."
4,DIFF_age_sq,32.093087,99,0.614474,0.032324,0.666794,0.042453,"[0.5756578947368421, 0.6019736842105263, 0.598..."
5,abs_centered_sq,32.093087,99,0.614474,0.026837,0.645204,0.035512,"[0.5888157894736842, 0.5822368421052632, 0.615..."


## Five Random 90/10 Evaluations

In [5]:
def make_model(random_state):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestClassifier(
                    **RF_PARAMS,
                    n_jobs=-1,
                    random_state=random_state,
                ),
            ),
        ]
    )


splitter = StratifiedShuffleSplit(
    n_splits=N_RANDOM_SPLITS,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

split_rows = []
last_model = None
last_test_idx = None
last_pred = None
last_proba = None

for split_num, (train_idx, test_idx) in enumerate(splitter.split(X, y), start=1):
    model = make_model(RANDOM_STATE + split_num)
    model.fit(X.iloc[train_idx], y.iloc[train_idx])

    pred = model.predict(X.iloc[test_idx])
    proba = model.predict_proba(X.iloc[test_idx])[:, 1]
    acc = accuracy_score(y.iloc[test_idx], pred)
    auc = roc_auc_score(y.iloc[test_idx], proba)

    split_rows.append({
        "split": split_num,
        "accuracy": acc,
        "roc_auc": auc,
        "train_rows": len(train_idx),
        "test_rows": len(test_idx),
    })

    last_model = model
    last_test_idx = test_idx
    last_pred = pred
    last_proba = proba

results = pd.DataFrame(split_rows)

print("Mean accuracy:", results["accuracy"].mean())
print("Accuracy std:", results["accuracy"].std(ddof=0))
print("Mean ROC AUC:", results["roc_auc"].mean())
print("ROC AUC std:", results["roc_auc"].std(ddof=0))
results

Mean accuracy: 0.6203947368421052
Accuracy std: 0.04589257901735603
Mean ROC AUC: 0.6773112880886426
ROC AUC std: 0.046621220820991176


,split,accuracy,roc_auc,train_rows,test_rows
0,1,0.562500,0.602969,2736,304
1,2,0.598684,0.672784,2736,304
2,3,0.592105,0.667633,2736,304
3,4,0.680921,0.747143,2736,304
4,5,0.667763,0.696027,2736,304


## Last Split Diagnostics

In [6]:
print("Confusion matrix for last split:")
print(confusion_matrix(y.iloc[last_test_idx], last_pred))
print("\nClassification report for last split:")
print(classification_report(y.iloc[last_test_idx], last_pred, target_names=["B", "R"]))

importances = pd.DataFrame({
    "feature": feature_cols,
    "importance": last_model.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

importances.head(30)

Confusion matrix for last split:
[[ 99  53]
 [ 48 104]]

Classification report for last split:
              precision    recall  f1-score   support

           B       0.67      0.65      0.66       152
           R       0.66      0.68      0.67       152

    accuracy                           0.67       304
   macro avg       0.67      0.67      0.67       304
weighted avg       0.67      0.67      0.67       304



,feature,importance
0,DIFF_age,0.034466
1,DIFF_age_sq,0.031532
2,DIFF_elo,0.023890
3,LOGDIFF_TOTAL_STR_absorbed_per_min,0.017781
4,DIFF_TOTAL_STR_absorbed_per_min,0.016520
5,DIFF_HEAD_absorbed_per_min,0.013465
6,LOGDIFF_HEAD_land_per_min,0.013140
7,LOGDIFF_GROUND_absorbed_per_min,0.012877
8,LOGDIFF_HEAD_absorbed_per_min,0.012875
9,DIFF_SIG_STR_acc,0.012661


## Train Final Model On All Rows

Use this model object if you want a final fitted RF after evaluating the split performance.

In [7]:
final_model = make_model(RANDOM_STATE)
final_model.fit(X, y)

final_importances = pd.DataFrame({
    "feature": feature_cols,
    "importance": final_model.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

final_importances.head(30)

,feature,importance
0,DIFF_age_sq,0.035474
1,DIFF_age,0.033769
2,DIFF_elo,0.021536
3,LOGDIFF_TOTAL_STR_absorbed_per_min,0.018225
4,DIFF_TOTAL_STR_absorbed_per_min,0.017722
5,DIFF_HEAD_absorbed_per_min,0.013657
6,DIFF_SIG_STR_acc,0.013510
7,LOGDIFF_HEAD_absorbed_per_min,0.013098
8,LOGDIFF_HEAD_land_per_min,0.012518
9,LOGDIFF_GROUND_opp_att_per_min,0.012359
